<div align="left" style="background-color: #008080; padding: 20px 10px;">
<h3><b>IDEAS - Institute of Data Engineering, Analytics and Science Foundation</b></h3>
<p>Summer Internship Program 2026</p>
<hr style="width:100%;">
<h3><b>Project Title:</b> Anomaly Detection for Fraud and Sensor Data</h3>
<h4>Project Notebook</h4>

<blockquote style="border-left: 4px solid #4285F4; padding-left: 15px;">
  <strong>Created by:</strong> Rounak Biswas<br>
  <strong>Designation:</strong> Project Linked Associate Research Engineer
</blockquote>
<hr style="width:100%;">
</div>

### Question 1: Load Libraries (2 Marks)

Import `numpy` as `np`, `pandas` as `pd`, `stats` from `scipy`, `IsolationForest` and `LocalOutlierFactor` from `sklearn.ensemble` and `sklearn.neighbors`, and `classification_report`, `precision_score`, `recall_score` from `sklearn.metrics`.

**Expected Output:** The code cell should execute without any errors.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import classification_report, precision_score, recall_score

### Question 2: Create the Dataset (4 Marks)

Generate a synthetic credit card transaction dataset. Set `np.random.seed(42)`. Create a DataFrame `normal` with 950 rows and a DataFrame `fraud` with 50 rows. Features should include `amount`, `hour_of_day`, `transactions_last_24h`, and `distance_from_home_km`, plus an `is_fraud` label (0 for normal, 1 for fraud). Combine them into a single DataFrame named `df`, shuffle using `.sample(frac=1, random_state=42)`, and reset the index.

**Hint:** Use `np.random.normal`, `np.random.randint`, `np.random.poisson`, and `np.random.exponential` to generate feature data.

**Expected Output:** Execution without errors, creating the `df` DataFrame.

In [ ]:
import numpy as np
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)

# Create normal transactions (950 records)
normal = pd.DataFrame({
    'amount': np.random.normal(loc=50, scale=20, size=950),
    'hour_of_day': np.random.randint(0, 24, 950),
    'transactions_last_24h': np.random.poisson(lam=3, size=950),
    'distance_from_home_km': np.random.exponential(scale=10, size=950),
    'is_fraud': 0})

# Create fraudulent transactions (50 records)
fraud = pd.DataFrame({
    'amount': np.random.normal(loc=300, scale=100, size=50),
    'hour_of_day': np.random.randint(0, 24, 50),
    'transactions_last_24h': np.random.poisson(lam=10, size=50),
    'distance_from_home_km': np.random.exponential(scale=50, size=50),
    'is_fraud': 1})

# Combine both datasets
df = pd.concat([normal, fraud], ignore_index=True)

# Shuffle dataset and reset index
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

### Question 3: Check Dataset Shape and Distribution (2 Marks)

Print the shape of your combined DataFrame `df` and the value counts of the `is_fraud` column to observe the class distribution.

**Expected Output:** The shape (1000, 5) and the counts showing 950 normal (0) and 50 fraud (1) cases.

In [ ]:
print(df.shape)
print(df['is_fraud'].value_counts())

(1000, 5)
is_fraud
0    950
1     50
Name: count, dtype: int64


### Question 4: Compare Feature Means by Class (3 Marks)

Group the dataset by the `is_fraud` column and calculate the mean values for all features (`amount`, `hour_of_day`, `transactions_last_24h`, `distance_from_home_km`). Print the resulting grouped means.

**Hint:** Use the `.groupby()` method and `.mean()`.

**Expected Output:** A table showing the mean feature values for normal (0) vs fraudulent (1) transactions.

In [ ]:
print(df.groupby('is_fraud')[[
    'amount',
    'hour_of_day',
    'transactions_last_24h',
    'distance_from_home_km']].mean())

              amount  hour_of_day  transactions_last_24h  \
is_fraud                                                   
0          50.401681    11.394737               2.984211   
1         301.755629    12.080000               9.940000   

          distance_from_home_km  
is_fraud                         
0                     10.071876  
1                     49.434432  


### Question 5: Apply Z-Score Anomaly Detection (3 Marks)

Compute the absolute Z-scores for the `distance_from_home_km` column. Create a new column named `zscore_anomaly` in `df` that contains `1` if the absolute Z-score is greater than 3, and `0` otherwise. Print the total number of flagged anomalies.

**Hint:** Use `np.abs(stats.zscore(...))`.

**Expected Output:** The count of anomalies flagged based on the distance feature.

In [ ]:
z_scores = np.abs(stats.zscore(df['distance_from_home_km']))

df['zscore_anomaly'] = (z_scores > 3).astype(int)

print(df['zscore_anomaly'].sum())

20


### Question 6: Apply IQR Method (4 Marks)

Calculate the Interquartile Range (IQR) for the `amount` column. Identify bounds: `Lower = Q1 - 1.5 * IQR` and `Upper = Q3 + 1.5 * IQR`. Create a new column named `iqr_anomaly` in `df` containing `1` for values outside these bounds and `0` otherwise. Print the total number of flagged anomalies.

**Hint:** Use `.quantile(0.25)` for Q1 and `.quantile(0.75)` for Q3.

**Expected Output:** The total number of `amount` anomalies flagged by the IQR method.

In [ ]:
Q1 = df['amount'].quantile(0.25)
Q3 = df['amount'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df['iqr_anomaly'] = ((df['amount'] < lower) | (df['amount'] > upper)).astype(int)

print(df['iqr_anomaly'].sum())

54


### Question 7: Train Isolation Forest (4 Marks)

Import `StandardScaler` from `sklearn.preprocessing` and scale the four feature columns. Then, create an `IsolationForest` model with `contamination=0.05` and `random_state=42`. Fit the model on the scaled features and add a column `isoforest_anomaly` to `df` containing `1` for anomalies and `0` for normal data.

**Hint:** `IsolationForest` returns `-1` for anomalies and `1` for normal points. Map these to `1` and `0` respectively.

**Expected Output:** The execution completes successfully.

In [ ]:
from sklearn.preprocessing import StandardScaler

features = ['amount', 'hour_of_day', 'transactions_last_24h', 'distance_from_home_km']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

iso = IsolationForest(contamination=0.05, random_state=42)

pred = iso.fit_predict(X_scaled)

df['isoforest_anomaly'] = (pred == -1).astype(int)



### Question 8: Evaluate Isolation Forest (3 Marks)

Use the `classification_report` function to evaluate the performance of your `isoforest_anomaly` predictions against the true `is_fraud` labels. Print the report.

**Expected Output:** A classification report displaying precision, recall, and f1-score for the model.

In [ ]:
print(classification_report(df['is_fraud'], df['isoforest_anomaly']))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       950
           1       0.92      0.92      0.92        50

    accuracy                           0.99      1000
   macro avg       0.96      0.96      0.96      1000
weighted avg       0.99      0.99      0.99      1000



### Question 9: Local Outlier Factor Detection (3 Marks)

Train a `LocalOutlierFactor` model with `n_neighbors=20` and `contamination=0.05` on the scaled features. Add a new column `lof_anomaly` to `df` (where `1` indicates an anomaly and `0` indicates normal).

**Hint:** Use `.fit_predict()` to get the anomaly flags (similar to Isolation Forest, LOF returns `-1` for anomalies).

**Expected Output:** The execution completes successfully.

In [30]:
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05
)

lof_pred = lof.fit_predict(X_scaled)

df['lof_anomaly'] = (lof_pred == -1).astype(int)

### Question 10: Evaluate LOF Model (2 Marks)

Calculate and print both the `precision_score` and `recall_score` for the `lof_anomaly` predictions against the true `is_fraud` labels.

**Expected Output:** Two numbers showing the precision and recall scores.

In [31]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score
from sklearn.neighbors import LocalOutlierFactor # Ensure LocalOutlierFactor is available

# Re-create X_scaled based on the current state of df
features = ['amount', 'hour_of_day', 'transactions_last_24h', 'distance_from_home_km']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

# Re-create lof_anomaly column as it appears to be missing
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05
)
lof_pred = lof.fit_predict(X_scaled)
df['lof_anomaly'] = (lof_pred == -1).astype(int)

print("Precision:", precision_score(df['is_fraud'], df['lof_anomaly']))
print("Recall:", recall_score(df['is_fraud'], df['lof_anomaly']))

Precision: 0.28
Recall: 0.28
